# 여러 채널 영상 메타데이터 수집

`CHANNEL_URLS`에 YouTube 채널 주소를 여러 개 넣으면 각 채널의 업로드 영상 메타데이터를 수집해서 `data/youtube/` 아래에 **채널별 DB 파일**로 저장합니다.

- 채널 URL은 `https://www.youtube.com/@handle/videos` 또는 `https://www.youtube.com/channel/UC...` 형태를 지원합니다.
- YouTube Data API key는 `.env`의 `YOUTUBE_API_KEY`를 사용합니다.
- 각 DB의 테이블명은 `youtube_videos`입니다.
- 파일명은 채널명과 채널 ID 일부를 조합해서 만듭니다.

예:

```text
data/youtube/un_trade_and_development_mjnwg_videos.db
data/youtube/international_trade_centre_tjmeag_videos.db
```


In [ ]:
import os
import re
import sqlite3
import time
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

API_KEY = os.getenv("YOUTUBE_API_KEY")
if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY not set. Check your .env file.")

CHANNEL_URLS = [
    "https://www.youtube.com/@UNCTADOnline/videos",
    "https://www.youtube.com/@InternationalTradeCentre/videos",
    "https://www.youtube.com/@FinancialTimes/videos"
]

DATA_DIR = Path("data/youtube")
TABLE_NAME = "youtube_videos"

MAX_PAGES_PER_CHANNEL = None  # 테스트할 때는 1~2, 전체 수집은 None
SLEEP_SECONDS = 0.1

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
}

DATA_DIR.mkdir(parents=True, exist_ok=True)


def chunked(values, size):
    for i in range(0, len(values), size):
        yield values[i:i + size]


def safe_filename(text, fallback="youtube_channel"):
    slug = re.sub(r"[^0-9a-zA-Z]+", "_", str(text)).strip("_").lower()
    slug = re.sub(r"_+", "_", slug)
    return slug or fallback


def channel_db_path(channel_info):
    channel_slug = safe_filename(channel_info["channel_name"], fallback=channel_info["channel_id"])
    channel_suffix = safe_filename(channel_info["channel_id"][-6:], fallback="channel")
    return DATA_DIR / f"{channel_slug}_{channel_suffix}_videos.db"


def resolve_channel_id(channel_url):
    parsed = urlparse(channel_url)
    parts = [part for part in parsed.path.split("/") if part]

    if len(parts) >= 2 and parts[0] == "channel" and parts[1].startswith("UC"):
        return parts[1]

    html = requests.get(channel_url, headers=HEADERS, timeout=30).text
    match = re.search(r'"externalId":"(UC[^"]+)"', html)
    if not match:
        raise ValueError(f"채널 ID를 찾지 못했습니다: {channel_url}")
    return match.group(1)


def fetch_channel_info(channel_id):
    res = requests.get(
        "https://www.googleapis.com/youtube/v3/channels",
        params={
            "part": "snippet,contentDetails,statistics",
            "id": channel_id,
            "key": API_KEY,
        },
        timeout=30,
    )
    res.raise_for_status()
    items = res.json().get("items", [])
    if not items:
        raise ValueError(f"채널 정보를 찾지 못했습니다: {channel_id}")

    item = items[0]
    return {
        "channel_id": channel_id,
        "channel_name": item["snippet"].get("title", ""),
        "channel_description": item["snippet"].get("description", ""),
        "channel_published_at": item["snippet"].get("publishedAt", ""),
        "subscriber_count": item.get("statistics", {}).get("subscriberCount", ""),
        "channel_video_count": item.get("statistics", {}).get("videoCount", ""),
        "uploads_playlist_id": item["contentDetails"]["relatedPlaylists"]["uploads"],
    }


def fetch_upload_playlist_items(channel_info, channel_url):
    rows = []
    next_page_token = None
    page_count = 0

    while True:
        params = {
            "part": "snippet,contentDetails",
            "playlistId": channel_info["uploads_playlist_id"],
            "maxResults": 50,
            "key": API_KEY,
        }
        if next_page_token:
            params["pageToken"] = next_page_token

        res = requests.get(
            "https://www.googleapis.com/youtube/v3/playlistItems",
            params=params,
            timeout=30,
        )
        res.raise_for_status()
        data = res.json()

        for item in data.get("items", []):
            snippet = item.get("snippet", {})
            video_id = snippet.get("resourceId", {}).get("videoId", "")
            if not video_id:
                continue

            rows.append({
                "source_channel_url": channel_url,
                "channel_id": channel_info["channel_id"],
                "channel_name": channel_info["channel_name"],
                "channel_description": channel_info["channel_description"],
                "channel_published_at": channel_info["channel_published_at"],
                "subscriber_count": channel_info["subscriber_count"],
                "channel_video_count": channel_info["channel_video_count"],
                "video_id": video_id,
                "title": snippet.get("title", ""),
                "description": snippet.get("description", ""),
                "published_at": snippet.get("publishedAt", ""),
                "url": f"https://www.youtube.com/watch?v={video_id}",
            })

        page_count += 1
        print(f"{channel_info['channel_name']} / {page_count}페이지 완료 / 누적 {len(rows)}개")

        next_page_token = data.get("nextPageToken")
        if not next_page_token:
            break
        if MAX_PAGES_PER_CHANNEL is not None and page_count >= MAX_PAGES_PER_CHANNEL:
            break

        time.sleep(SLEEP_SECONDS)

    return rows


def fetch_video_details(video_ids):
    detail_rows = []
    for ids in tqdm(list(chunked(video_ids, 50)), desc="video details"):
        res = requests.get(
            "https://www.googleapis.com/youtube/v3/videos",
            params={
                "part": "snippet,contentDetails,statistics,status",
                "id": ",".join(ids),
                "key": API_KEY,
            },
            timeout=30,
        )
        res.raise_for_status()

        for item in res.json().get("items", []):
            stats = item.get("statistics", {})
            details = item.get("contentDetails", {})
            status = item.get("status", {})
            snippet = item.get("snippet", {})
            detail_rows.append({
                "video_id": item.get("id", ""),
                "duration": details.get("duration", ""),
                "caption_available": details.get("caption", ""),
                "definition": details.get("definition", ""),
                "dimension": details.get("dimension", ""),
                "view_count": stats.get("viewCount", ""),
                "like_count": stats.get("likeCount", ""),
                "comment_count": stats.get("commentCount", ""),
                "privacy_status": status.get("privacyStatus", ""),
                "made_for_kids": status.get("madeForKids", ""),
                "category_id": snippet.get("categoryId", ""),
                "tags": ", ".join(snippet.get("tags", [])),
                "default_language": snippet.get("defaultLanguage", ""),
                "default_audio_language": snippet.get("defaultAudioLanguage", ""),
            })
        time.sleep(SLEEP_SECONDS)

    return pd.DataFrame(detail_rows)


saved_db_paths = []

for channel_url in CHANNEL_URLS:
    print("채널 처리 시작:", channel_url)
    channel_id = resolve_channel_id(channel_url)
    channel_info = fetch_channel_info(channel_id)
    channel_rows = fetch_upload_playlist_items(channel_info, channel_url)

    videos_df = pd.DataFrame(channel_rows).drop_duplicates("video_id", keep="last").reset_index(drop=True)
    print(f"[{channel_info['channel_name']}] 수집된 영상 수:", len(videos_df))

    if len(videos_df):
        detail_df = fetch_video_details(videos_df["video_id"].tolist())
        videos_df = videos_df.merge(detail_df, on="video_id", how="left")

    db_path = channel_db_path(channel_info)
    conn = sqlite3.connect(db_path)
    videos_df.to_sql(TABLE_NAME, conn, if_exists="replace", index=False)
    conn.close()

    saved_db_paths.append(db_path)
    print("채널별 DB 저장 완료:", db_path)
    print("table:", TABLE_NAME)
    print("rows:", len(videos_df))
    print()

print("저장된 DB 목록:")
for db_path in saved_db_paths:
    print("-", db_path)


# 채널별 영상 자막 수집 후 DB 저장

`data/youtube/*_videos.db` 형태의 채널별 메타데이터 DB를 하나씩 읽고, `youtube-transcript-api`로 자막을 가져와 새 컬럼으로 저장합니다.

실행 전 설치:

```python
%pip install youtube-transcript-api
```

각 입력 DB마다 결과 DB를 따로 만듭니다.

예:

```text
data/youtube/un_trade_and_development_mjnwg_videos.db
→ data/youtube/un_trade_and_development_mjnwg_videos_with_transcripts.db

data/youtube/international_trade_centre_tjmeag_videos.db
→ data/youtube/international_trade_centre_tjmeag_videos_with_transcripts.db
```

테이블명은 모두 `youtube_videos`입니다.


In [ ]:
import sqlite3
import time
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from youtube_transcript_api import YouTubeTranscriptApi

DATA_DIR = Path("data/youtube")
TABLE_NAME = "youtube_videos"
SAVE_EVERY = 50
SLEEP_SECONDS = 0.2
TRANSCRIPT_LANGUAGES = ["en", "en-US", "ko"]

# 채널별 메타데이터 DB만 읽습니다.
# 예: un_trade_and_development_mjnwg_videos.db
CHANNEL_VIDEO_DB_PATHS = sorted(
    path for path in DATA_DIR.glob("*_videos.db")
    if not path.name.endswith("_with_transcripts.db")
    and path.name != "youtube_videos.db"
)

if not CHANNEL_VIDEO_DB_PATHS:
    raise FileNotFoundError("data/youtube 안에 채널별 *_videos.db 파일이 없습니다.")

print("처리할 채널별 DB 목록:")
for db_path in CHANNEL_VIDEO_DB_PATHS:
    print("-", db_path)


def transcript_items_to_text(items):
    texts = []
    for item in items:
        text = item.get("text", "") if isinstance(item, dict) else getattr(item, "text", "")
        text = str(text).replace("\n", " ").strip()
        if text:
            texts.append(text)
    return " ".join(texts)


def fetch_transcript(video_id, languages=TRANSCRIPT_LANGUAGES):
    # youtube-transcript-api 버전에 따라 static API와 instance API가 달라서 둘 다 지원합니다.
    try:
        items = YouTubeTranscriptApi.get_transcript(video_id, languages=languages)
        return transcript_items_to_text(items), len(items), ""
    except AttributeError:
        api = YouTubeTranscriptApi()
        fetched = api.fetch(video_id, languages=languages)
        items = fetched.to_raw_data() if hasattr(fetched, "to_raw_data") else fetched
        language = getattr(fetched, "language_code", "")
        return transcript_items_to_text(items), len(items), language


def transcript_db_path(video_db_path):
    return video_db_path.with_name(f"{video_db_path.stem}_with_transcripts.db")


def read_table(db_path):
    conn = sqlite3.connect(db_path)
    df = pd.read_sql(f"SELECT * FROM {TABLE_NAME}", conn)
    conn.close()
    return df


def load_channel_dataframe(video_db_path, output_db_path):
    if output_db_path.exists():
        print("기존 자막 DB에서 이어서 실행합니다:", output_db_path)
        return read_table(output_db_path)
    return read_table(video_db_path)


def save_channel_dataframe(df, output_db_path):
    output_db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(output_db_path)
    df.to_sql(TABLE_NAME, conn, if_exists="replace", index=False)
    conn.close()


for video_db_path in CHANNEL_VIDEO_DB_PATHS:
    output_db_path = transcript_db_path(video_db_path)
    df = load_channel_dataframe(video_db_path, output_db_path)

    transcript_cols = [
        "transcript_text",
        "transcript_line_count",
        "transcript_language",
        "transcript_status",
        "transcript_error",
    ]
    for col in transcript_cols:
        if col not in df.columns:
            df[col] = ""
        # SQLite/pyarrow가 문자열 컬럼으로 읽어도 숫자 대입에서 터지지 않게 object로 둡니다.
        df[col] = df[col].astype("object")

    transcript_filled = df["transcript_text"].notna() & df["transcript_text"].astype(str).str.strip().ne("")
    video_id_filled = df["video_id"].notna() & df["video_id"].astype(str).str.strip().ne("")
    target_indices = df.index[video_id_filled & ~transcript_filled].tolist()

    print("\n입력 DB:", video_db_path)
    print("저장 DB:", output_db_path)
    print("전체 행 수:", len(df))
    print("자막 수집 대상:", len(target_indices))

    processed = 0
    for idx in tqdm(target_indices, desc=f"{video_db_path.stem} transcripts"):
        video_id = str(df.at[idx, "video_id"]).strip()
        try:
            transcript_text, line_count, language = fetch_transcript(video_id)
            df.at[idx, "transcript_text"] = transcript_text
            df.at[idx, "transcript_line_count"] = str(line_count)
            df.at[idx, "transcript_language"] = language
            df.at[idx, "transcript_status"] = "success"
            df.at[idx, "transcript_error"] = ""
        except Exception as e:
            df.at[idx, "transcript_text"] = ""
            df.at[idx, "transcript_line_count"] = "0"
            df.at[idx, "transcript_language"] = ""
            df.at[idx, "transcript_status"] = "failed"
            df.at[idx, "transcript_error"] = f"{type(e).__name__}: {e}"
            print(f"[skip] {video_id} ({type(e).__name__}: {e})")

        processed += 1
        if processed % SAVE_EVERY == 0:
            save_channel_dataframe(df, output_db_path)
            print(f"중간 저장 완료: {processed}/{len(target_indices)}")

        time.sleep(SLEEP_SECONDS)

    save_channel_dataframe(df, output_db_path)
    print("최종 저장 완료:", output_db_path)
    print(df["transcript_status"].value_counts(dropna=False))

# Selenium 웹크롤링 방식 자막 보완 수집

`youtube-transcript-api`가 `IpBlocked` 등으로 막히는 경우를 대비해, YouTube 페이지를 직접 열고 `스크립트 표시 / Show transcript` 패널에서 자막 텍스트를 읽어옵니다.

- 입력: `data/youtube/*_videos_with_transcripts.db`가 있으면 우선 사용하고, 없으면 `*_videos.db`를 사용합니다.
- 이미 `transcript_text`가 있는 행은 건너뜁니다.
- 결과는 기존 `*_videos_with_transcripts.db`에 다시 저장합니다.
- 실행 전 필요하면 설치하세요.

```python
%pip install selenium webdriver-manager
```


In [ ]:
import sqlite3
import time
from pathlib import Path

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from tqdm import tqdm

DATA_DIR = Path("data/youtube")
TABLE_NAME = "youtube_videos"
SAVE_EVERY = 10
PAGE_SLEEP_SECONDS = 3
BETWEEN_VIDEO_SLEEP_SECONDS = 1
WAIT_SECONDS = 1
HEADLESS = False
MAX_VIDEOS_PER_DB = 5  # 테스트할 때는 5~10, 전체 수집은 None


def get_channel_video_db_paths():
    base_paths = sorted(
        path for path in DATA_DIR.glob("*_videos.db")
        if not path.name.endswith("_with_transcripts.db")
        and path.name != "youtube_videos.db"
    )
    result = []
    for base_path in base_paths:
        transcript_path = base_path.with_name(f"{base_path.stem}_with_transcripts.db")
        result.append((base_path, transcript_path if transcript_path.exists() else base_path))
    return result


def output_db_path(input_or_transcript_path):
    if input_or_transcript_path.name.endswith("_with_transcripts.db"):
        return input_or_transcript_path
    return input_or_transcript_path.with_name(f"{input_or_transcript_path.stem}_with_transcripts.db")


def read_table(db_path):
    conn = sqlite3.connect(db_path)
    df = pd.read_sql(f"SELECT * FROM {TABLE_NAME}", conn)
    conn.close()
    return df


def save_table(df, db_path):
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)
    df.to_sql(TABLE_NAME, conn, if_exists="replace", index=False)
    conn.close()


def safe_click(driver, elem):
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", elem)
    time.sleep(0.2)
    driver.execute_script("arguments[0].click();", elem)


def click_first_matching_button(driver, keywords):
    buttons = driver.find_elements(By.CSS_SELECTOR, "button, tp-yt-paper-button")
    for button in buttons:
        text = (button.get_attribute("textContent") or "").strip().lower()
        aria = (button.get_attribute("aria-label") or "").strip().lower()
        merged = f"{text} {aria}"
        if any(keyword.lower() in merged for keyword in keywords):
            safe_click(driver, button)
            return True
    return False


def expand_description_if_possible(driver):
    selectors = [
        "tp-yt-paper-button#expand",
        "#expand",
        "#more",
        "ytd-text-inline-expander tp-yt-paper-button",
    ]
    for selector in selectors:
        for elem in driver.find_elements(By.CSS_SELECTOR, selector):
            try:
                safe_click(driver, elem)
                return True
            except Exception:
                pass

    try:
        return click_first_matching_button(driver, ["더보기", "show more", "more"])
    except Exception:
        return False


def open_transcript_panel(driver, wait):
    for _ in range(3):
        expand_description_if_possible(driver)

        transcript_selectors = [
            "button[aria-label='스크립트 표시']",
            "button[aria-label='Show transcript']",
            "button[aria-label='Open transcript']",
        ]
        for selector in transcript_selectors:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            if elems:
                safe_click(driver, elems[0])
                return True

        if click_first_matching_button(driver, ["스크립트", "transcript"]):
            return True

        driver.execute_script("window.scrollBy(0, 700);")
        time.sleep(0.8)

    xpath = "//button[contains(@aria-label, '스크립트') or contains(@aria-label, 'transcript') or contains(., '스크립트') or contains(., 'Transcript')]"
    elem = wait.until(EC.presence_of_element_located((By.XPATH, xpath)))
    safe_click(driver, elem)
    return True


def collect_transcript_text(driver, wait):
    selectors = [
        "yt-formatted-string.segment-text",
        "ytd-transcript-segment-renderer .segment-text",
        "ytd-transcript-segment-renderer yt-formatted-string",
        "[class*='segment-text']",
    ]
    for selector in selectors:
        try:
            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector)))
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            texts = [elem.get_attribute("textContent").strip() for elem in elems]
            texts = [text for text in texts if text]
            if texts:
                return " ".join(texts), len(texts)
        except Exception:
            pass
    return "", 0


def fetch_transcript_by_web(driver, url):
    wait = WebDriverWait(driver, WAIT_SECONDS)
    driver.get(url)
    time.sleep(PAGE_SLEEP_SECONDS)
    open_transcript_panel(driver, wait)
    time.sleep(1)
    text, line_count = collect_transcript_text(driver, wait)
    if not text:
        raise ValueError("웹페이지에서 transcript 텍스트를 찾지 못했습니다.")
    return text, line_count


def make_driver():
    opts = webdriver.ChromeOptions()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--lang=ko-KR")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)


db_pairs = get_channel_video_db_paths()
if not db_pairs:
    raise FileNotFoundError("data/youtube 안에 채널별 *_videos.db 파일이 없습니다.")

print("처리할 DB 목록:")
for base_path, input_path in db_pairs:
    print("-", input_path)

for base_path, input_path in db_pairs:
    out_path = output_db_path(input_path)
    df = read_table(input_path)

    for col in [
        "transcript_text",
        "transcript_line_count",
        "transcript_language",
        "transcript_status",
        "transcript_error",
        "transcript_source",
    ]:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].astype("object")

    transcript_filled = df["transcript_text"].notna() & df["transcript_text"].astype(str).str.strip().ne("")
    url_filled = df["url"].notna() & df["url"].astype(str).str.strip().ne("")
    target_indices = df.index[url_filled & ~transcript_filled].tolist()
    if MAX_VIDEOS_PER_DB is not None:
        target_indices = target_indices[:MAX_VIDEOS_PER_DB]

    print("\n입력 DB:", input_path)
    print("저장 DB:", out_path)
    print("전체 행 수:", len(df))
    print("웹크롤링 자막 수집 대상:", len(target_indices))

    if not target_indices:
        save_table(df, out_path)
        continue

    driver = make_driver()
    processed = 0

    try:
        for idx in tqdm(target_indices, desc=f"{out_path.stem} web transcripts"):
            url = str(df.at[idx, "url"]).strip()
            try:
                transcript_text, line_count = fetch_transcript_by_web(driver, url)
                df.at[idx, "transcript_text"] = transcript_text
                df.at[idx, "transcript_line_count"] = str(line_count)
                df.at[idx, "transcript_language"] = ""
                df.at[idx, "transcript_status"] = "success"
                df.at[idx, "transcript_error"] = ""
                df.at[idx, "transcript_source"] = "selenium_web"
            except Exception as e:
                df.at[idx, "transcript_text"] = ""
                df.at[idx, "transcript_line_count"] = "0"
                df.at[idx, "transcript_language"] = ""
                df.at[idx, "transcript_status"] = "failed_web"
                df.at[idx, "transcript_error"] = f"{type(e).__name__}: {e}"
                df.at[idx, "transcript_source"] = "selenium_web"
                print(f"[skip-web] {url} ({type(e).__name__}: {e})")

            processed += 1
            if processed % SAVE_EVERY == 0:
                save_table(df, out_path)
                print(f"중간 저장 완료: {processed}/{len(target_indices)}")

            time.sleep(BETWEEN_VIDEO_SLEEP_SECONDS)

    finally:
        save_table(df, out_path)
        driver.quit()
        print("저장 완료:", out_path)
        print(df["transcript_status"].value_counts(dropna=False))


처리할 DB 목록:
- data/youtube/financial_times_5rsvpw_videos_with_transcripts.db
- data/youtube/international_trade_centre_tjmeag_videos.db
- data/youtube/un_trade_and_development_mjnwg_videos.db

입력 DB: data/youtube/financial_times_5rsvpw_videos_with_transcripts.db
저장 DB: data/youtube/financial_times_5rsvpw_videos_with_transcripts.db
전체 행 수: 11321
웹크롤링 자막 수집 대상: 5


financial_times_5rsvpw_videos_with_transcripts web transcripts:   0%|          | 0/5 [00:00<?, ?it/s]

[skip-web] https://www.youtube.com/watch?v=gZXBh5yDGqs (TimeoutException: Message: 
Stacktrace:
0   chromedriver                        0x0000000100b6f1e4 cxxbridge1$str$ptr + 3175992
1   chromedriver                        0x0000000100b672ac cxxbridge1$str$ptr + 3143424
2   chromedriver                        0x0000000100617374 _RNvCs9NA52FrZ4pF_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74080
3   chromedriver                        0x000000010065f964 _RNvCs9NA52FrZ4pF_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 370512
4   chromedriver                        0x000000010069ef4c _RNvCs9NA52FrZ4pF_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 630072
5   chromedriver                        0x0000000100655354 _RNvCs9NA52FrZ4pF_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 328000
6   chromedriver                        0x0000000100b2d370 cxxbridge1$str$ptr + 2906052
7   chromedriver                        0x0000000100b30af0 cxxbridge1$str$ptr + 2920260
8   chromedriver 

financial_times_5rsvpw_videos_with_transcripts web transcripts:  20%|██        | 1/5 [00:18<01:13, 18.48s/it]

[skip-web] https://www.youtube.com/watch?v=kmp8yQCPTBI (ValueError: 웹페이지에서 transcript 텍스트를 찾지 못했습니다.)


...

un_trade_and_development_mjnwg_videos_with_transcripts web transcripts:  60%|██████    | 3/5 [00:06<00:03,  1.84s/it]

[skip-web] https://www.youtube.com/watch?v=79s-wrgAw8I (NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=149.0.7827.197)
Stacktrace:
0   chromedriver                        0x00000001048d71e4 cxxbridge1$str$ptr + 3175992
1   chromedriver                        0x00000001048cf2ac cxxbridge1$str$ptr + 3143424
2   chromedriver                        0x000000010437f374 _RNvCs9NA52FrZ4pF_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74080
3   chromedriver                        0x000000010435874c chromedriver + 165708
4   chromedriver                        0x00000001043efa74 _RNvCs9NA52FrZ4pF_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 534624
5   chromedriver                        0x0000000104406a68 _RNvCs9NA52FrZ4pF_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 628820
6   chromedriver                        0x00000001043bd354 _RNvCs9NA52FrZ4pF_7___rustc35___rust_no_alloc_shim_is_unsta

un_trade_and_development_mjnwg_videos_with_transcripts web transcripts:  60%|██████    | 3/5 [00:07<00:05,  2.52s/it]

저장 완료: data/youtube/un_trade_and_development_mjnwg_videos_with_transcripts.db
transcript_status
              1716
failed_web       4
Name: count, dtype: int64


KeyboardInterrupt: 